# XLS-R full fine-tuning
based on https://huggingface.co/blog/fine-tune-xlsr-wav2vec2


In [1]:
%pip install -U pip
%pip install --no-cache-dir 'transformers==4.57.1' accelerate 'datasets[audio]' evaluate jiwer safetensors huggingface_hub tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 67.0 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 236.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 136.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 796.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 1.2 GB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.2 GB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 253.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 590.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 192.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 296.6 MB/s  0:00:00
  Attempting uninstall: hugg

In [1]:
import os
import re
import json
import gc
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Union

import numpy as np
import pandas as pd
import torch
import jiwer

from tqdm.auto import tqdm
from datasets import load_dataset, Audio
from huggingface_hub import notebook_login, HfFolder, create_repo

from transformers import (Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor,
    Wav2Vec2ForCTC, TrainingArguments, Trainer, EarlyStoppingCallback, set_seed)

In [2]:
notebook_login()

In [2]:
hf_token = HfFolder.get_token()

if hf_token is None:
    raise ValueError('HF token was not found. Run notebook_login() first.')

In [3]:
def create_xlsr_model(model_id):
    model = Wav2Vec2ForCTC.from_pretrained(
        model_id,
        attention_dropout=0.0,
        hidden_dropout=0.0,
        feat_proj_dropout=0.0,
        mask_time_prob=0.05,
        layerdrop=0.0,
        ctc_loss_reduction='mean',
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer),
        ignore_mismatched_sizes=True
    )

    model.freeze_feature_encoder()

    return model


def extract_all_chars(batch):
    all_text = ' '.join(batch['sentence'])
    vocab = sorted(list(set(all_text)))
    return {'vocab': [vocab], 'all_text': [all_text]}


def prepare_dataset(batch):
    audio = batch['audio']

    batch['input_values'] = processor(audio['array'], sampling_rate=audio['sampling_rate']).input_values[0]
    batch['input_length'] = len(batch['input_values'])
    batch['labels'] = processor(text=batch['sentence']).input_ids

    return batch


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_values': feature['input_values']} for feature in features]
        label_features = [{'input_ids': feature['labels']} for feature in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors='pt')
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors='pt')

        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch['labels'] = labels

        return batch


def normalize_spaces(text):
    if text is None:
        return ''

    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)

    label_str = processor.batch_decode(label_ids, group_tokens=False)

    pred_str = [normalize_spaces(text) for text in pred_str]
    label_str = [normalize_spaces(text) for text in label_str]

    wer = jiwer.wer(label_str, pred_str)
    cer = jiwer.cer(label_str, pred_str)

    return {'wer': wer, 'cer': cer}


def print_trainable_parameters(model):
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    all_params = sum(param.numel() for param in model.parameters())

    print(f'Trainable params: {trainable_params:,}')
    print(f'All params: {all_params:,}')
    print(f'Trainable share: {100 * trainable_params / all_params:.4f}%')


def get_best_dev_metrics(trainer, log_history_df):
    best_checkpoint = trainer.state.best_model_checkpoint
    best_dev_cer = trainer.state.best_metric

    best_step = None
    best_dev_loss = None
    best_dev_wer = None

    if best_checkpoint is not None:
        match = re.search(r'checkpoint-(\d+)', best_checkpoint)

        if match is not None:
            best_step = int(match.group(1))

            best_eval_rows = log_history_df[
                (log_history_df['step'] == best_step) &
                (log_history_df['eval_cer'].notna())
            ]

            if len(best_eval_rows) > 0:
                best_eval_row = best_eval_rows.iloc[0]
                best_dev_loss = best_eval_row['eval_loss']
                best_dev_wer = best_eval_row['eval_wer']
                best_dev_cer = best_eval_row['eval_cer']

    return {
        'best_dev_checkpoint': best_checkpoint,
        'best_dev_step': best_step,
        'best_dev_loss': best_dev_loss,
        'best_dev_WER': best_dev_wer,
        'best_dev_CER': best_dev_cer
    }


def prepare_resource_test_dataset(resource):
    test_dataset_raw_resource = dataset_dict['test'].filter(lambda example: example['resource'] == resource)

    if len(test_dataset_raw_resource) == 0:
        raise ValueError(f'No test examples found for resource: {resource}')

    test_dataset_resource = test_dataset_raw_resource.map(prepare_dataset,
        remove_columns=test_dataset_raw_resource.column_names, load_from_cache_file=False)

    decoded_labels = processor.batch_decode(test_dataset_resource['labels'], group_tokens=False)
    decoded_labels = [normalize_spaces(text) for text in decoded_labels]

    references = [normalize_spaces(text) for text in test_dataset_raw_resource['sentence']]
    mismatches = [(i, ref, label) for i, (ref, label) in enumerate(zip(references, decoded_labels)) if ref != label]

    print(f'{resource} mismatches before predict:', len(mismatches))

    if len(mismatches) > 0:
        print(mismatches[:5])
        raise ValueError(f'{resource}: raw references and prepared labels do not match')

    return test_dataset_raw_resource, test_dataset_resource


def predict_dataset_in_order(model, prepared_dataset, raw_dataset, data_collator, processor, batch_size=8):
    device = next(model.parameters()).device
    model.eval()

    rows = []

    for start in tqdm(range(0, len(prepared_dataset), batch_size)):
        end = min(start + batch_size, len(prepared_dataset))

        features = [prepared_dataset[i] for i in range(start, end)]

        batch = data_collator(features)

        input_batch = {key: value.to(device) for key, value in batch.items() if key != 'labels'}

        with torch.no_grad():
            logits = model(**input_batch).logits

        pred_ids = torch.argmax(logits, dim=-1)

        pred_str = processor.batch_decode(pred_ids)
        pred_str = [normalize_spaces(text) for text in pred_str]

        for local_i, prediction in enumerate(pred_str):
            raw_i = start + local_i
            raw_example = raw_dataset[raw_i]

            rows.append({
                'resource': raw_example['resource'],
                'path': raw_example['path'],
                'reference': normalize_spaces(raw_example['sentence']),
                'prediction': prediction
            })

    return pd.DataFrame(rows)


def evaluate_resource_test(resource, experiment_name, model, batch_size=8):
    test_dataset_raw_resource, test_dataset_resource = prepare_resource_test_dataset(resource)

    results_df = predict_dataset_in_order(model=model, prepared_dataset=test_dataset_resource,
        raw_dataset=test_dataset_raw_resource, data_collator=data_collator,
        processor=processor, batch_size=batch_size)

    wer = jiwer.wer(results_df['reference'].tolist(), results_df['prediction'].tolist())

    cer = jiwer.cer(results_df['reference'].tolist(), results_df['prediction'].tolist())

    predictions_path = f'{experiment_name}_{resource}_test_predictions.csv'

    results_df.to_csv(predictions_path, index=False, encoding='utf-8-sig')

    print(f'{resource}_test WER:', wer)
    print(f'{resource}_test CER:', cer)

    row = {
        'model': model_label,
        'training': experiment_name,
        'subset': f'{resource}_test',
        'WER': wer,
        'CER': cer,
        'n_files': len(results_df),
        'predictions_file': predictions_path
    }

    return row, results_df

In [4]:
dataset_repo_id = 'tadgeis/chukchi-asr-data-private'

dataset_dict = load_dataset(dataset_repo_id, token=hf_token)
dataset_dict = dataset_dict.cast_column('audio', Audio(sampling_rate=16_000))

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 2714
    })
    test: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 429
    })
    dev: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 140
    })
})

In [5]:
train_chars = set(' '.join(dataset_dict['train']['sentence']))
dev_chars = set(' '.join(dataset_dict['dev']['sentence']))
test_chars = set(' '.join(dataset_dict['test']['sentence']))

print('Chars in dev but not train:', dev_chars - train_chars)
print('Chars in test but not train:', test_chars - train_chars)

print('Train chars:', sorted(train_chars))

Chars in dev but not train: set()
Chars in test but not train: set()
Train chars: [' ', "'", 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ъ', 'ы', 'ь', 'э', 'ю', 'я', 'ё', 'ӄ', 'ӈ', 'ԓ']


In [6]:
vocab_source_dataset = dataset_dict['train']

vocab_train = vocab_source_dataset.map(extract_all_chars, batched=True, batch_size=-1,
    keep_in_memory=True, remove_columns=vocab_source_dataset.column_names)

vocab_list = vocab_train['vocab'][0]
vocab_dict = {char: idx for idx, char in enumerate(vocab_list)}

vocab_dict['|'] = vocab_dict[' ']
del vocab_dict[' ']

vocab_dict['[UNK]'] = len(vocab_dict)
vocab_dict['[PAD]'] = len(vocab_dict)

Map:   0%|          | 0/2714 [00:00<?, ? examples/s]

In [7]:
with open('vocab.json', 'w', encoding='utf-8') as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False, indent=2)

In [8]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained('./', unk_token='[UNK]', pad_token='[PAD]', word_delimiter_token='|')

In [9]:
sample_text = dataset_dict['train'][0]['sentence']

encoded = tokenizer(sample_text).input_ids
decoded = tokenizer.decode(encoded, group_tokens=False)

print('Original:', sample_text)
print('Encoded:', encoded)
print('Decoded:', decoded)

Original: ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты
Encoded: [29, 14, 29, 37, 30, 16, 0, 12, 31, 13, 10, 20, 0, 20, 29, 36, 10, 4, 29, 13, 35, 29, 13, 20, 10, 0, 35, 13, 10, 12, 12, 10, 15, 0, 33, 15, 4, 2, 18, 30, 20, 2, 5, 15, 31, 20, 29]
Decoded: ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты


In [10]:
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16_000,
    padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [11]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# fine-tuning on chuklang only

In [12]:
experiment_name = 'xlsr_1b_chuklang_only'
model_id = 'facebook/wav2vec2-xls-r-1b'
model_repo_id = 'tadgeis/xlsr-1b-ckt-chuklang-only'
model_label = 'XLS-R 1B CTC fine-tuning'

resources_to_evaluate = ['chuklang', 'radio', 'bible']

In [13]:
create_repo(repo_id=model_repo_id, repo_type='model', private=False, exist_ok=True, token=hf_token)
processor.push_to_hub(model_repo_id, private=False, token=hf_token)

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/tadgeis/xlsr-1b-ckt-chuklang-only/commit/9e35ca55baba161520afd74280f8fe9e71855557', commit_message='Upload processor', commit_description='', oid='9e35ca55baba161520afd74280f8fe9e71855557', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tadgeis/xlsr-1b-ckt-chuklang-only', endpoint='https://huggingface.co', repo_type='model', repo_id='tadgeis/xlsr-1b-ckt-chuklang-only'), pr_revision=None, pr_num=None)

In [14]:
SEED = 42
set_seed(SEED)

train_dataset_raw = dataset_dict['train'].filter(lambda example: example['resource'] == 'chuklang')
eval_dataset_raw = dataset_dict['dev'].filter(lambda example: example['resource'] == 'chuklang')

train_dataset_raw = train_dataset_raw.shuffle(seed=SEED)

print('Train raw:', train_dataset_raw)
print('Eval raw:', eval_dataset_raw)

Train raw: Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 659
})
Eval raw: Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 140
})


In [15]:
train_dataset = train_dataset_raw.map(prepare_dataset, remove_columns=train_dataset_raw.column_names, load_from_cache_file=False)
eval_dataset = eval_dataset_raw.map(prepare_dataset, remove_columns=eval_dataset_raw.column_names, load_from_cache_file=False)

print('Prepared train:', train_dataset)
print('Prepared eval:', eval_dataset)
print('Train columns:', train_dataset.column_names)
print('Eval columns:', eval_dataset.column_names)

Map: 100%|##########| 659/659 [00:00<?, ? examples/s]

Map: 100%|##########| 140/140 [00:00<?, ? examples/s]

Prepared train: Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 659
})
Prepared eval: Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 140
})
Train columns: ['input_values', 'input_length', 'labels']
Eval columns: ['input_values', 'input_length', 'labels']


In [16]:
model = create_xlsr_model(model_id)

print_trainable_parameters(model)

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-1b and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 958,341,034
All params: 962,551,210
Trainable share: 99.5626%


In [17]:
# # test on the longest audio files

# def test_longest_batch(model, train_dataset, data_collator, batch_size=4):
#     if not torch.cuda.is_available():
#         raise RuntimeError('CUDA is not available.')

#     model.to('cuda')
#     model.train()
#     model.zero_grad(set_to_none=True)

#     outputs = None
#     loss = None
#     batch = None
#     features = None

#     longest_indices = np.argsort(
#         -np.array(train_dataset['input_length'])
#     )[:batch_size]

#     print('Testing indices:', longest_indices.tolist())
#     print(
#         'Lengths in seconds:',
#         [
#             round(train_dataset[int(i)]['input_length'] / 16000, 2)
#             for i in longest_indices
#         ]
#     )

#     try:
#         features = [
#             train_dataset[int(i)]
#             for i in longest_indices
#         ]

#         batch = data_collator(features)

#         batch = {
#             key: value.to('cuda')
#             for key, value in batch.items()
#         }

#         with torch.autocast(
#             device_type='cuda',
#             dtype=torch.float16
#         ):
#             outputs = model(**batch)
#             loss = outputs.loss

#         loss.backward()

#         print('Loss:', loss.item())
#         print('Longest-batch train forward/backward: PASSED')

#         return True

#     except torch.cuda.OutOfMemoryError as error:
#         print('CUDA OOM: batch_size is too large for this GPU/model/audio length.')
#         print(error)

#         return False

#     finally:
#         model.zero_grad(set_to_none=True)

#         del batch
#         del features
#         del outputs
#         del loss

#         gc.collect()

#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.ipc_collect()

#         !nvidia-smi


# test_longest_batch(model=model, train_dataset=train_dataset, data_collator=data_collator, batch_size=16) ## batch size

In [18]:
training_args = TrainingArguments(
    output_dir=model_repo_id.split('/')[-1],

    group_by_length=True,

    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=16,

    eval_strategy='steps',
    save_strategy='steps',

    num_train_epochs=36,

    gradient_checkpointing=False,
    fp16=torch.cuda.is_available(),

    save_steps=50,
    eval_steps=50,
    logging_steps=50,

    learning_rate=1e-5, ## for 1B parameters
    warmup_steps=25,

    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,

    push_to_hub=True,
    hub_model_id=model_repo_id,
    hub_private_repo=False,
    hub_token=hf_token,
    hub_strategy='checkpoint',
    hub_always_push=True,

    report_to='none',
    disable_tqdm=False,

    seed=SEED,
    data_seed=SEED
)

In [19]:
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor.feature_extractor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=6,
            early_stopping_threshold=0.001
        )
    ]
)

In [20]:
!rm -rf xlsr-1b-ckt-chuklang-only/checkpoint-*

In [21]:
!df -h
!du -h --max-depth=1 . | sort -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         200G   14G  187G   7% /
tmpfs            64M     0   64M   0% /dev
shm             157G  4.0K  157G   1% /dev/shm
/dev/md0        7.0T  1.9T  5.2T  27% /etc/hosts
tmpfs           126G  1.6M  126G   1% /run/nvidia-persistenced/socket
/dev/root       124G   11G  114G   9% /usr/bin/nvidia-smi
tmpfs           315G     0  315G   0% /proc/acpi
tmpfs           315G     0  315G   0% /proc/scsi
tmpfs           315G     0  315G   0% /sys/firmware
du: cannot read directory './proc/945/task/945/fdinfo': Permission denied
du: cannot read directory './proc/945/map_files': Permission denied
du: cannot read directory './proc/945/fdinfo': Permission denied
du: cannot read directory './proc/966/task/966/fdinfo': Permission denied
du: cannot read directory './proc/966/map_files': Permission denied
du: cannot read directory './proc/966/fdinfo': Permission denied
du: cannot read directory './proc/1063/task/1063/fdinfo': Permission denied
du: 

In [22]:
trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
50,8.873200,3.168415,1.000000,1.000000
100,3.021600,2.773105,1.000000,1.000000
150,1.980900,2.198575,1.000000,0.915127
200,1.314000,1.809993,1.000000,0.725626
250,0.989500,1.585793,1.000000,0.576550
300,0.793800,1.559135,1.000000,0.539601
350,0.705100,1.433599,0.996825,0.452533
400,0.589700,1.422286,0.990476,0.413755
450,0.499700,1.338130,0.988889,0.375343
500,0.419500,1.378481,0.976190,0.376258


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


TrainOutput(global_step=1450, training_loss=0.8262579648248081, metrics={'train_runtime': 1033.3949, 'train_samples_per_second': 22.957, 'train_steps_per_second': 1.463, 'total_flos': 1.2214939335650763e+19, 'train_loss': 0.8262579648248081, 'epoch': 34.523809523809526})

In [23]:
log_history_df = pd.DataFrame(trainer.state.log_history)

log_history_df.to_csv(f'{experiment_name}_log_history.csv', index=False, encoding='utf-8-sig')

best_metrics = get_best_dev_metrics(trainer, log_history_df)

print(best_metrics)

{'best_dev_checkpoint': 'xlsr-1b-ckt-chuklang-only/checkpoint-1150', 'best_dev_step': 1150, 'best_dev_loss': np.float64(1.3706690073013306), 'best_dev_WER': np.float64(0.9634920634920635), 'best_dev_CER': np.float64(0.3307115419791476)}


In [24]:
processor.save_pretrained(training_args.output_dir)
trainer.save_model(training_args.output_dir)

trainer.push_to_hub()

processor.push_to_hub(model_repo_id, private=False, token=hf_token)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/tadgeis/xlsr-1b-ckt-chuklang-only/commit/928899c2854115ff94da156b7f0c8b8eef690fc5', commit_message='Upload processor', commit_description='', oid='928899c2854115ff94da156b7f0c8b8eef690fc5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tadgeis/xlsr-1b-ckt-chuklang-only', endpoint='https://huggingface.co', repo_type='model', repo_id='tadgeis/xlsr-1b-ckt-chuklang-only'), pr_revision=None, pr_num=None)

In [25]:
summary_rows = []
prediction_dfs = {}

for resource in resources_to_evaluate:
    row, results_df = evaluate_resource_test(resource=resource, experiment_name=experiment_name,
        model=trainer.model, batch_size=16) ### batch size

    row['best_dev_checkpoint'] = best_metrics['best_dev_checkpoint']
    row['best_dev_step'] = best_metrics['best_dev_step']
    row['best_dev_loss'] = best_metrics['best_dev_loss']
    row['best_dev_WER'] = best_metrics['best_dev_WER']
    row['best_dev_CER'] = best_metrics['best_dev_CER']

    summary_rows.append(row)
    prediction_dfs[resource] = results_df

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(f'{experiment_name}_results_summary.csv', index=False, encoding='utf-8-sig')

summary_df

Filter:   0%|          | 0/429 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

chuklang mismatches before predict: 0


  0%|          | 0/13 [00:00<?, ?it/s]

chuklang_test WER: 0.9706227967097533
chuklang_test CER: 0.3332873118873395


...

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

bible mismatches before predict: 0


  0%|          | 0/10 [00:00<?, ?it/s]

bible_test WER: 0.9822590183323477
bible_test CER: 0.3106168809086581


,model,training,subset,WER,CER,n_files,predictions_file,best_dev_checkpoint,best_dev_step,best_dev_loss,best_dev_WER,best_dev_CER
0,XLS-R 1B CTC fine-tuning,xlsr_1b_chuklang_only,chuklang_test,0.970623,0.333287,200,xlsr_1b_chuklang_only_chuklang_test_prediction...,xlsr-1b-ckt-chuklang-only/checkpoint-1150,1150,1.370669,0.963492,0.330712
1,XLS-R 1B CTC fine-tuning,xlsr_1b_chuklang_only,radio_test,0.998665,0.440494,83,xlsr_1b_chuklang_only_radio_test_predictions.csv,xlsr-1b-ckt-chuklang-only/checkpoint-1150,1150,1.370669,0.963492,0.330712
2,XLS-R 1B CTC fine-tuning,xlsr_1b_chuklang_only,bible_test,0.982259,0.310617,146,xlsr_1b_chuklang_only_bible_test_predictions.csv,xlsr-1b-ckt-chuklang-only/checkpoint-1150,1150,1.370669,0.963492,0.330712


In [29]:
prediction_dfs['chuklang'].head()

,resource,path,reference,prediction
0,chuklang,A chatterbox and a wanton girl_2.wav,ӄоле итгъэт ӄынвэтэ ӈиръэ ӈэвысӄэтти элерэты н...,ӄоле итгъат ӄнвытэ ӈиръэӈэвчӄатэлйирэтынатанат
1,chuklang,A chatterbox and a wanton girl_3.wav,ӄол вэтгавӈавъым ӄол камэлгыӈав,ӄолывэтгъавӈаъм ӄолкамэлгиӈа
2,chuklang,A chatterbox and a wanton girl_4.wav,ынкы илирыкы нантыӈӈонатъым,нкыилирк антыӈӈонатъым
3,chuklang,Abramovich_4.wav,гэчевкы нынтыӄин таӈколё ынкы ныгынритӄин,эчевкнынтӄитаӈколмквын ыгнрит
4,chuklang,An evil spirit and a dicky bird_1.wav,энмэн гатвален каԓьайӈын ынкъам пчеӄалгын,энмэы гатвалын кальъаэйынкъампчэкали


In [26]:
files_to_download = [Path(f'{experiment_name}_log_history.csv'),
                     Path(f'{experiment_name}_results_summary.csv')]

for resource in resources_to_evaluate:
    files_to_download.append(Path(f'{experiment_name}_{resource}_test_predictions.csv'))

for path in files_to_download:
    if path.exists():
        print(f'Ready to download: {path}')
    else:
        print(f'File not found: {path}')

Ready to download: xlsr_1b_chuklang_only_log_history.csv
Ready to download: xlsr_1b_chuklang_only_results_summary.csv
Ready to download: xlsr_1b_chuklang_only_chuklang_test_predictions.csv
Ready to download: xlsr_1b_chuklang_only_radio_test_predictions.csv
Ready to download: xlsr_1b_chuklang_only_bible_test_predictions.csv
